In [1]:
from dotenv import find_dotenv, load_dotenv
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from transformers import AutoTokenizer

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
CHROMA_STORAGE_PATH = "../data/vectors/chroma"

In [3]:
load_dotenv(find_dotenv("../../creds/.env"), verbose=True)

True

In [4]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
)

embedding = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

vector_db = Chroma(
    embedding_function=embedding,
    persist_directory=CHROMA_STORAGE_PATH,
)
vector_db._collection.count()

721

In [5]:
question = "Analyze the materials and answer this question: what is the major topic for this class?"
docs = vector_db.similarity_search(
    query=question,
    # filter={"source": "../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf"},
    k=3,
)
docs

[Document(id='37a0c17a-896f-429d-b0a2-f79d79c69fd9', metadata={'title': '', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:24:59-07:00', 'total_pages': 20, 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'page_label': '1', 'page': 0, 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture04.pdf', 'creationdate': '2008-07-11T11:24:59-07:00', 'author': ''}, page_content='MachineLearning-Lecture04  \nInstructor (Andrew Ng):Okay, good morning. Just a few administrative announcements \nbefore we jump into today’s technical material. So let’s see, by later today, I’ll post on \nthe course website a handout with the sort of guidelines and suggestions for choosing and \nproposing class projects.  \nSo project proposals – so for the term project for this class due on Friday, the 19th of this \nmonth at noon – that’s about two weeks, two and a half weeks from now. If you haven’t \nyet formed teams or started thinking about project ideas, please do so.  \nAnd later toda

In [6]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever(),
)

In [7]:
result = qa_chain.invoke({"query": question})
print(result["result"].strip())

The major topic for this class is **machine learning**. The instructor discusses project proposals and examples (e.g., neuroscience, fMRI data, financial trading, facial attractiveness, optical illusions) that align with applying machine learning techniques. The term project emphasizes using machine learning to solve interesting problems, as noted in the lecture.


### Prompt

In [8]:
template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum. Keep the answer as concise as possible.
Always say "thanks for asking!" at the end of the answer.
{context}

Question: {question}

Helpful Answer:
"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
QA_CHAIN_PROMPT

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nIf you don\'t know the answer, just say that you don\'t know, don\'t try to make up an answer.\nUse three sentences maximum. Keep the answer as concise as possible.\nAlways say "thanks for asking!" at the end of the answer.\n{context}\n\nQuestion: {question}\n\nHelpful Answer:\n')

In [9]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": QA_CHAIN_PROMPT,
    },
)

In [10]:
question = "Is probability a class topic?"
result = qa_chain.invoke({"query": question})
print(result["result"].strip())
result

Yes, probability is a key topic in this class, covering foundations and applications like empirical risk minimization. The course includes discussions on probability and statistics prerequisites. Thanks for asking!


{'query': 'Is probability a class topic?',
 'result': '\n\nYes, probability is a key topic in this class, covering foundations and applications like empirical risk minimization. The course includes discussions on probability and statistics prerequisites. Thanks for asking!',
 'source_documents': [Document(id='805814dd-af7f-4000-9ce9-dcb931f3321f', metadata={'creator': 'PScript5.dll Version 5.2.2', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture04.pdf', 'title': '', 'total_pages': 20, 'page_label': '2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'author': '', 'moddate': '2008-07-11T11:24:59-07:00', 'page': 1, 'creationdate': '2008-07-11T11:24:59-07:00'}, page_content='So throughout the previous lecture and this lecture, we’re starting to use increasingly \nlarge amounts of material on probability. So if you’d like to see a refresher on sort of the \nfoundations of probability – if you’re not sure if you quite had your prerequisites for this \nclass in terms of a back

### RetrievalQA chain types

In [11]:
qa_chain_mr = RetrievalQA.from_chain_type(
    llm,
    retriever=vector_db.as_retriever(),
    chain_type="map_reduce",
)

result = qa_chain_mr.invoke({"query": question})
print(result["result"].strip())
result

Yes, the text explicitly mentions probability as a class topic. Relevant excerpts include:  
- "we’re starting to use increasingly large amounts of material on probability."  
- "the discussion section taught this week by the TA’s will go over so they can review a probability."  
- "if you’re not sure if you quite had your prerequisites for this class in terms of a background in probability and statistics..."  

These passages confirm that probability is a foundational topic in the class.


{'query': 'Is probability a class topic?',
 'result': '\n\nYes, the text explicitly mentions probability as a class topic. Relevant excerpts include:  \n- "we’re starting to use increasingly large amounts of material on probability."  \n- "the discussion section taught this week by the TA’s will go over so they can review a probability."  \n- "if you’re not sure if you quite had your prerequisites for this class in terms of a background in probability and statistics..."  \n\nThese passages confirm that probability is a foundational topic in the class.'}

In [13]:
qa_chain_refine = RetrievalQA.from_chain_type(
    llm,
    retriever=vector_db.as_retriever(),
    chain_type="refine",
)

result = qa_chain_refine.invoke({"query": question})
print(result["result"].strip())
result

The original answer is accurate but can be refined to better align with the context provided. Here's the updated version:

Probability is a central topic in this class, particularly in the context of **empirical risk minimization (ERM)** and its analysis. The instructor’s discussion highlights that ERM is a general framework for learning, where the hypothesis class $ \mathcal{H} $ can include any set of functions (e.g., linear classifiers, logistic regression, or more complex models). The key to understanding whether ERM is a reasonable algorithm lies in **probability theory**, which provides bounds on the generalization error. For example, **Hoeffding’s inequality** (or similar probabilistic tools) quantifies how likely it is that the training error differs significantly from the generalization error, ensuring that ERM selects hypotheses that are not only optimal on the training data but also likely to perform well on unseen data. This probabilistic analysis is critical for justifying

{'query': 'Is probability a class topic?',
 'result': "\n\nThe original answer is accurate but can be refined to better align with the context provided. Here's the updated version:\n\nProbability is a central topic in this class, particularly in the context of **empirical risk minimization (ERM)** and its analysis. The instructor’s discussion highlights that ERM is a general framework for learning, where the hypothesis class $ \\mathcal{H} $ can include any set of functions (e.g., linear classifiers, logistic regression, or more complex models). The key to understanding whether ERM is a reasonable algorithm lies in **probability theory**, which provides bounds on the generalization error. For example, **Hoeffding’s inequality** (or similar probabilistic tools) quantifies how likely it is that the training error differs significantly from the generalization error, ensuring that ERM selects hypotheses that are not only optimal on the training data but also likely to perform well on unsee

### RetrievalQA limitations

In [14]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vector_db.as_retriever(),
)

In [15]:
question = "Is probability a class topic?"
result = qa_chain.invoke({"query": question})
result["result"].strip()

'Yes, probability is a class topic. The context explicitly mentions that the course covers foundational probability and statistics prerequisites, with discussion sections dedicated to reviewing these topics. The instructor also references probability in the context of logistic regression and empirical risk minimization, indicating that probability is a key component of the curriculum.'

In [19]:
question = "what is it needed for?"
result = qa_chain.invoke({"query": question})
result["result"].strip()

'The context discusses the importance of understanding how to use machine learning tools like support vector machines (SVMs) effectively, rather than just memorizing theory or math. The "objective function" (W of Alpha) mentioned is part of solving the optimization problem for SVMs, which aims to find the optimal hyperplane for classification. This function is needed to determine the best parameters (e.g., alpha values) that maximize the margin between classes while minimizing misclassifications. The key takeaway is that true mastery of machine learning involves knowing how to adapt and troubleshoot models, not just applying formulas.'